# 13. Scientific Toolkit — `scipy`
**Python Libraries Lab · Phase 1 · Data Science**

_Statistical tests, optimisation, interpolation and more._

✅ **Runs in Colab** as-is.

NumPy gives you arrays. SciPy gives you the scientific algorithms that work on them. Statistics and tests, finding the best value of something, filling in missing points, integration, signal processing and more. You rarely use all of it, but a few parts are gold for everyday analysis.

**In this notebook:** You will run that A B test, check whether ratings differ between cities, find the price that maximises profit with an optimiser, and fill in missing daily temperatures with interpolation.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q scipy
import importlib.metadata as md
for pkg in ['scipy']:
    print(pkg, md.version(pkg))

## Step 1 · Load the ShopEasy dataset
Same 600 orders as the pandas lesson, created fresh so this notebook runs on its own.

In [ ]:
import numpy as np, pandas as pd

rng = np.random.default_rng(42)
n = 600
orders = pd.DataFrame({
    "order_id": np.arange(1001, 1001 + n),
    "date": pd.Timestamp("2026-01-01") + pd.to_timedelta(rng.integers(0, 90, n), unit="D"),
    "city": rng.choice(["Mumbai", "Delhi", "Pune", "Bengaluru"], n),
    "category": rng.choice(["Books", "Electronics", "Clothing", "Home"], n, p=[0.3, 0.2, 0.3, 0.2]),
    "quantity": rng.integers(1, 6, n),
    "price": rng.choice([199, 499, 999, 1499, 2999], n),
    "rating": rng.choice([1, 2, 3, 4, 5], n, p=[0.05, 0.1, 0.2, 0.35, 0.3]),
})
orders["revenue"] = orders["quantity"] * orders["price"]
orders.to_csv("shop_orders.csv", index=False)
print(orders.shape, "-> saved shop_orders.csv")
orders.head()

## Step 2 · A/B test with a t-test
Two groups of order values. `ttest_ind` compares their means.

In [ ]:
from scipy import stats

rng = np.random.default_rng(7)
old_page = rng.normal(1500, 400, 200)      # average order value, old checkout
new_page = rng.normal(1600, 400, 200)      # new checkout

t, p = stats.ttest_ind(new_page, old_page)
print(f"Old mean {old_page.mean():.0f} | New mean {new_page.mean():.0f}")
print(f"t = {t:.2f}, p-value = {p:.4f}")
print("Real difference" if p < 0.05 else "Could be luck")

## Step 3 · Compare many groups (ANOVA)
Do average ratings differ across the four cities?

In [ ]:
groups = [g["rating"].values for _, g in orders.groupby("city")]
f, p = stats.f_oneway(*groups)
print(orders.groupby("city")["rating"].mean().round(2))
print(f"\nANOVA p-value = {p:.3f} ->", "cities differ" if p < 0.05 else "no clear difference")

## Step 4 · Describe and use distributions
`stats.norm` is the bell curve. Ask what share of values fall below a point.

In [ ]:
print(stats.describe(orders["revenue"]))
mu, sigma = 1500, 400
print(f"\nShare of orders under Rs.1000: {stats.norm.cdf(1000, mu, sigma):.1%}")
print(f"Top 5% of orders are above:   Rs.{stats.norm.ppf(0.95, mu, sigma):.0f}")

## Step 5 · Optimise: find the best price
Demand falls as price rises. `minimize_scalar` finds the price with the most profit (we minimise negative profit).

In [ ]:
from scipy.optimize import minimize_scalar

def profit(price, cost=300):
    demand = 1000 * np.exp(-price / 800)     # fewer buyers at higher prices
    return (price - cost) * demand

best = minimize_scalar(lambda p: -profit(p), bounds=(300, 3000), method="bounded")
print(f"Best price: Rs.{best.x:.0f}  ->  profit Rs.{profit(best.x):,.0f}")

## Step 6 · Interpolate missing points
We only measured warehouse temperature every 3 hours. `interp1d` fills the gaps.

In [ ]:
from scipy.interpolate import interp1d

hours = np.array([0, 3, 6, 9, 12])
temps = np.array([22, 20, 24, 30, 33])
f = interp1d(hours, temps, kind="cubic")
for h in [1, 4.5, 10]:
    print(f"{h:>4}h -> {f(h):.1f} C")

## ✍️ Your turn
Run a t-test comparing `a` and `b` below. Store the **p-value** in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
a = [12, 14, 15, 13, 16, 14]
b = [18, 17, 19, 20, 18, 21]
answer = None  # your code here

In [ ]:
def check():
    expected = stats.ttest_ind([12, 14, 15, 13, 16, 14], [18, 17, 19, 20, 18, 21]).pvalue
    ok = answer is not None and abs(float(answer) - expected) < 1e-9
    return ok, f"Expected about {expected:.6f}, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
a = [12, 14, 15, 13, 16, 14]
b = [18, 17, 19, 20, 18, 21]
answer = stats.ttest_ind(a, b).pvalue

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **Hypothesis test**
- **p-value**
- **t-test**
- **Optimisation**
- **Interpolation**
- **Distribution**

**Next:** 14. Polars (`polars`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/14_polars.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install scipy
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.